# watermark removal testing on Kaggle (2x T4)

Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**.
Inputs: the `wmr-0.2.1.zip` dataset and your video dataset. Run the cells in order.

In [ ]:
import glob, json, os, shutil, zipfile
VERSION = '0.2.1'
ROOT = f'/kaggle/working/wmr-{VERSION}'

def find_wmr():
    for base, dirs, files in os.walk('/kaggle/input', followlinks=True):
        if f'wmr-{VERSION}.zip' in files:
            return 'zip', os.path.join(base, f'wmr-{VERSION}.zip')
        if os.path.basename(base) == 'scripts' and 'setup_kaggle.sh' in files and \
                os.path.exists(os.path.join(os.path.dirname(base), 'wmr', 'track.py')):
            init = open(os.path.join(os.path.dirname(base), 'wmr', '__init__.py')).read()
            if VERSION in init:
                return 'dir', os.path.dirname(base)
    if os.path.exists(f'/kaggle/working/wmr-{VERSION}.zip'):
        return 'zip', f'/kaggle/working/wmr-{VERSION}.zip'
    return None, None

if not os.path.exists(ROOT):
    kind, path = find_wmr()
    print(kind, path)
    if kind == 'zip':
        zipfile.ZipFile(path).extractall('/kaggle/working')
    elif kind == 'dir':
        shutil.copytree(path, ROOT)
    else:
        raise SystemExit(f'wmr {VERSION} not found in /kaggle/input: add the wmr-{VERSION}.zip dataset as an input')
print(sorted(os.listdir(ROOT)))

In [ ]:
!bash /kaggle/working/wmr-0.2.1/scripts/setup_kaggle.sh

## 1. Self-test on a synthetic clip

`frames_with_mask_recall_below_95pct` should be 0, `psnr_in_mask_after` well above `psnr_in_mask_before`,
and the run should report 3 tracks (one per overlay).

In [ ]:
%cd /kaggle/working
!python -m wmr synth -o synth1 --size 1280x720 --frames 300 --overlay "@my_channel:bounce:1.0:40" --overlay "SAMPLE MARK:static:0.4:40" --overlay "hello2026:teleport:1.0:40" --dwell 100
!python -m wmr run synth1/marked.mp4 -o synth1/out.mp4 --texts synth1/texts.txt --preview synth1/preview.mp4
!python -m wmr eval synth1 synth1/out.mp4 --tracks synth1/out.tracks.json

## 2. Your videos

`IN_DIR` is your dataset folder. If it does not exist, the cell looks for the first folder of videos under `/kaggle/input`.
Leave `TEXTS` empty to rely on discovery only, or list the overlay strings you know, one per line.

In [ ]:
IN_DIR  = '/kaggle/input/datasets/yxyxtx/video-test-set-1'   # folder with your videos
OUT_DIR = '/kaggle/working/out'
TEXTS = ''

VIDEO_EXTS = ('.mp4', '.mov', '.mkv', '.m4v', '.webm', '.avi')
if not os.path.isdir(IN_DIR):
    IN_DIR = next((b for b, d, f in os.walk('/kaggle/input', followlinks=True)
                   if 'wmr' not in b and any(x.lower().endswith(VIDEO_EXTS) for x in f)), None)
print('IN_DIR =', IN_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
open('/kaggle/working/texts.txt', 'w').write(TEXTS.strip() + '\n')
videos = sorted(f for f in os.listdir(IN_DIR) if f.lower().endswith(VIDEO_EXTS))
print(len(videos), 'videos:', videos[:20])

### One video first: analyze and look at the preview

Red boxes will be removed. Yellow boxes were discovered but need your approval: set `"enabled": true` for that track
in `/kaggle/working/one.tracks.json` (or `false` on anything you want to keep), then run the next cell.

In [ ]:
VIDEO = os.path.join(IN_DIR, videos[0])
!python -m wmr analyze "{VIDEO}" -o /kaggle/working/one.tracks.json --texts /kaggle/working/texts.txt --discover --preview /kaggle/working/one_preview.mp4
from IPython.display import Video
Video('/kaggle/working/one_preview.mp4', embed=True, width=720)

In [ ]:
!python -m wmr remove "{VIDEO}" /kaggle/working/one.tracks.json -o /kaggle/working/one_out.mp4
!python -m wmr verify /kaggle/working/one_out.mp4 --tracks /kaggle/working/one.tracks.json
Video('/kaggle/working/one_out.mp4', embed=True, width=720)

### The whole folder, one worker per GPU

Each video gets `<name>_clean.mp4`, a tracks file, a report and a log in `OUT_DIR`.
Discovered tracks that need approval are skipped, so check the summary's track list.

In [ ]:
!python -m wmr batch "{IN_DIR}" -o "{OUT_DIR}" --texts /kaggle/working/texts.txt --discover --gpus 0,1
for f in sorted(glob.glob(OUT_DIR + '/*.report.json')):
    r = json.load(open(f)); v = r.get('verify', {})
    tr = json.load(open(r['tracks']))['tracks']
    removed = [t['label'] for t in tr if t['enabled']]
    review = [t['label'] for t in tr if not t['enabled']]
    print(os.path.basename(r['output']), '| removed:', removed, '| needs review:', review,
          '| leaked frames:', v.get('leak_frames'), 'of', v.get('checked_frames'))

## 3. Optional: ProPainter engine for static opaque overlays

Higher quality where the background under the text is never visible; uses both GPUs for a single video.
ProPainter is under the NTU S-Lab License 1.0 (non-commercial). Not yet run by the author: a failed job falls back to the
built-in fill and leaves an `error.log` next to the job.

In [ ]:
!bash /kaggle/working/wmr-0.2.1/scripts/setup_kaggle.sh propainter
!python -m wmr remove "{VIDEO}" /kaggle/working/one.tracks.json -o /kaggle/working/one_out_pp.mp4 --engine propainter --gpus 0,1 --keep-work
Video('/kaggle/working/one_out_pp.mp4', embed=True, width=720)